In [3]:
import symlib
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import warnings; warnings.simplefilter('ignore')
import sys
import h5py
import pandas as pd
import seaborn as sns
sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/mcmc/src/')
import jsm_ancillary
import jsm_visualize
import jsm_SHMR
import jsm_mcmc
import jsm_stats
import jsm_models
import evolve as ev
import galhalo as gh
import profiles as profiles
import config as cfg

In [5]:
plt.style.use('../../../SatGen/notebooks/paper1/paper.mplstyle')
double_textwidth = 7.0 #inches
single_textwidth = 3.5 #inches
levelz = [1-0.99, 1-0.95, 1-0.68]

### claude written code

In [51]:
class Symphony_SubhaloCounter:
    """
    Measure subhalo statistics above a mass threshold within R_vir
    for each host halo in a symlib simulation suite.

    Parameters
    ----------
    base_dir       : base directory of the simulation suite (str)
    suite_name     : name of the simulation suite (str)
    mass_threshold : minimum subhalo mass to count (float, default 1e9 M_sun)

    Attributes
    ----------
    df : pandas DataFrame with columns:
        logMvir   - log10 host virial mass at z=0
        log1pz50  - log10(1 + z_50), where z_50 is the half-mass redshift
        logc      - log10 host concentration at z=0
        logMs     - log10 mass of most massive surviving subhalo within R_vir
        logNsub   - log10 number of subhalos within R_vir above threshold
        logfsub   - log10 subhalo mass fraction (Ms_tot / Mvir)
    """

    def __init__(self, base_dir, suite_name, mass_threshold=1e9):
        self.base_dir        = base_dir
        self.suite_name      = suite_name
        self.mass_threshold  = mass_threshold
        self.n_hosts         = symlib.n_hosts(suite_name)

        self.df = self._build()

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------
    def _find_nearest(self, array, value):
        return (np.abs(array - value)).argmin()

    def _process_host(self, i_host):
        """
        Process a single host halo and return a dict of statistics,
        or None if the host should be skipped.
        """
        sim_dir = symlib.get_host_directory(self.base_dir, self.suite_name, i_host)

        r, hist  = symlib.read_rockstar(sim_dir)
        s, hist  = symlib.read_symfind(sim_dir)

        h, hist2 = symlib.read_subhalos(sim_dir)
        scale    = symlib.scale_factors(sim_dir)

        # ------------------------------------------------------------------
        # Host halo properties
        # ------------------------------------------------------------------
        ok_host   = r[0, :]["ok"]
        a         = scale[ok_host]
        Mhost     = r[0, ok_host]["m"]
        Mhost_z0  = Mhost[-1]
        chost_z0  = r[0, ok_host]["cvir"][-1]
        host_rvir = r[0, -1]["rvir"]

        # half-mass redshift z50
        a50 = a[self._find_nearest(Mhost, Mhost_z0 / 2)]
        z50 = 1. / a50 - 1.

        # ------------------------------------------------------------------
        # Subhalo selection: within R_vir, alive at z=0, above mass threshold
        # ------------------------------------------------------------------
        sub_x = h[:, -1]["x"]
        r_3d  = np.sqrt(np.sum(sub_x**2, axis=1))
        ok    = h["ok"][:, -1]

        msub_all = np.array([r[i, -1]["m"] for i in range(1, len(r))])

        within_rvir  = r_3d[1:] < host_rvir
        above_thresh = msub_all > self.mass_threshold
        ok_subs      = ok[1:] & within_rvir & above_thresh

        msub_selected = msub_all[ok_subs]

        N_sub = len(msub_selected)
        if N_sub == 0:
            return None

        M_max  = np.max(msub_selected)
        M_sub  = np.sum(msub_selected)
        f_sub  = M_sub / Mhost_z0

        return {
            'logMvir' : np.log10(Mhost_z0),
            'log1pz50': np.log10(1 + z50),
            'logc'    : np.log10(chost_z0),
            'logMs'   : np.log10(M_max),
            'logNsub' : np.log10(N_sub),
            'logfsub' : np.log10(f_sub),
        }

    def _build(self):
        """Loop over all hosts and collect results into a DataFrame."""
        rows = []
        for i_host in range(self.n_hosts):
            try:
                row = self._process_host(i_host)
                if row is not None:
                    rows.append(row)
            except Exception as e:
                print(f"Error processing i_host={i_host}: {e}")
                continue

        return pd.DataFrame(rows, columns=['logMvir', 'log1pz50', 'logc',
                                           'logMs',   'logNsub',  'logfsub'])


In [62]:
test_fid = Symphony_SubhaloCounter("/Users/jsmonzon/Research/Misc/Symphony/", "SymphonyMilkyWay", 1.2*10**8)

In [63]:
test_500 = Symphony_SubhaloCounter("/Users/jsmonzon/Research/Misc/Symphony/", "SymphonyMilkyWay", 500*((1.2*10**8)/300))

In [64]:
test_1000 = Symphony_SubhaloCounter("/Users/jsmonzon/Research/Misc/Symphony/", "SymphonyMilkyWay", 1000*((1.2*10**8)/300))

In [66]:
rho1_fid = jsm_stats.correlation(test_fid.df["log1pz50"], test_fid.df["logNsub"])
rho2_fid = jsm_stats.correlation(test_fid.df["logc"], test_fid.df["logNsub"])
rho3_fid = jsm_stats.correlation(test_fid.df["logMs"], test_fid.df["logNsub"])

In [67]:
rho1_500 = jsm_stats.correlation(test_500.df["log1pz50"], test_500.df["logNsub"])
rho2_500 = jsm_stats.correlation(test_500.df["logc"], test_500.df["logNsub"])
rho3_500 = jsm_stats.correlation(test_500.df["logMs"], test_500.df["logNsub"])

In [68]:
rho1_1000 = jsm_stats.correlation(test_1000.df["log1pz50"], test_1000.df["logNsub"])
rho2_1000 = jsm_stats.correlation(test_1000.df["logc"], test_1000.df["logNsub"])
rho3_1000 = jsm_stats.correlation(test_1000.df["logMs"], test_1000.df["logNsub"])

In [71]:
symrho_arr = np.array([[rho1_fid, rho2_fid, rho3_fid], [rho1_500, rho2_500, rho3_500], [rho1_1000, rho2_1000, rho3_1000]])

In [80]:
symrho_arr[1]

array([-0.58678952, -0.58117104,  0.61718096])